# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the Kilifi County dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print("Keywords:", ', '.join(metadata.keywords))
print("Use cases:", ', '.join(metadata.dataUseCases))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @id
print("Record Sets and Fields Available:\n")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else 'N/A'}")
    print("  Fields:")
    for f in rs.fields:
        print(f"    - {f.name} (@id: {f.id}) [{f.data_type}]")
    print()

# Preview a few records from the main record set (choose the main one)
main_record_set = record_sets[0]
for i, record in enumerate(dataset.records(record_set=main_record_set.id)):
    print(f"Record #{i+1}: {record}")
    if i >= 2:
        break

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Print columns from the main record set
print("Columns in main record set:", dataframes[main_record_set.id].columns.tolist())
dataframes[main_record_set.id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# Example: Filter by 'phq9_score' and normalize
# Use field @id for 'phq9_score' and 'age' if available.

# Identify relevant numeric fields and group field by their @id
# You may adapt these values based on the schema overview above
phq9_field = None
age_field = None
education_field = None

# Find field @id by searching
for f in main_record_set.fields:
    if f.name.lower() == 'phq9_score':
        phq9_field = f.id
    if f.name.lower() == 'age':
        age_field = f.id
    if f.name.lower() == 'level_of_education':
        education_field = f.id

# EDA with PHQ-9 scores
df = dataframes[main_record_set.id]
if phq9_field in df.columns:
    threshold = 10
    filtered_df = df[df[phq9_field] > threshold]
    print(f"Filtered records with {phq9_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize PHQ-9 scores
    filtered_df[f"{phq9_field}_normalized"] = (filtered_df[phq9_field] - filtered_df[phq9_field].mean()) / filtered_df[phq9_field].std()
    print(f"Normalized {phq9_field} for filtered records:")
    print(filtered_df[[phq9_field, f"{phq9_field}_normalized"].head()])

    # Group by level_of_education if available
    grouping_field = education_field if education_field in df.columns else None
    if grouping_field:
        grouped_df = filtered_df.groupby(grouping_field)[phq9_field].mean().reset_index()
        print(f"Grouped mean {phq9_field} by {grouping_field}:")
        print(grouped_df.head())
else:
    print("PHQ-9 score field not found in DataFrame columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of PHQ-9 scores (referenced by @id)
if phq9_field and phq9_field in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[phq9_field], bins=20, kde=True)
    plt.title(f"Distribution of PHQ-9 Score ({phq9_field})")
    plt.xlabel("PHQ-9 Score")
    plt.ylabel("Count")
    plt.show()

# Boxplot of PHQ-9 scores grouped by education (if available)
if phq9_field and education_field and education_field in df.columns:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=df[education_field], y=df[phq9_field])
    plt.title(f"PHQ-9 Scores by Level of Education ({education_field})")
    plt.xlabel("Level of Education")
    plt.ylabel("PHQ-9 Score")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The Kilifi County dataset provides granular survey records on mental health indicators, with structured fields referenced by their unique `@id`s.
- Using `mlcroissant`, we loaded metadata, explored available fields and record sets, filtered and normalized PHQ-9 scores, and visualized aggregated results.
- Analysis of PHQ-9 scores, especially grouped by level of education, can inform targeted community health initiatives and public health strategies.


<!-- End of notebook -->